# Inside the Model: Regularisation

## Which coefficients should the model choose?

**35 minutes**

Ridge and Lasso balance prediction error against coefficient size. When two predictors carry the same information, what determines which coefficient gets the credit?

### Learning objectives

- distinguish the data-fit term from the coefficient penalty;
- explain why Ridge and Lasso can prefer different coefficient allocations;
- predict how changing a feature's units changes its penalty.

**Prerequisites:** linear regression, MSE, and the Ridge and Lasso penalties. The formulas needed below are supplied.

Write each prediction before calculating. Begin Python after the manual comparisons.


## 1. Compare fit and penalty

Consider three equally weighted observations:

| Input x | Response y |
|--:|--:|
| −1 | −2 |
| 0 | 0 |
| 1 | 2 |

First compare two proposed lines, $\hat y=bx$, with slopes 1 and 2. The intercept is **fixed at zero**. Ridge balances a data-fit term and a penalty:

$$D=\frac13\sum_{i=1}^{3}(y_i-\hat y_i)^2,\qquad
J_R=D+\lambda b^2,\qquad \lambda=\frac43.$$

Here $D$ is ordinary MSE. Keep this definition and the same $\lambda$ throughout the exercise.

Before calculating, predict which proposed line Ridge prefers. Does choosing between them require more than knowing which fits the data exactly?

**Your prediction:** _Write here._

The data-fit terms are supplied. Complete the penalty and total columns.

| Slope b | Data-fit term D | Penalty | Total objective |
|--:|--:|--:|--:|
| 1 | 2/3 | … | … |
| 2 | 0 | … | … |

Explain what the preferred candidate gives up and what it gains. This comparison chooses between **two candidates**; it does not find the best slope among all real numbers.

**Your calculation and explanation:** _Write here._


## 2. Two predictors, one signal

Now use two identical predictor columns: $x_1=x$ and $x_2=x$. Keep the same responses and zero intercept.

$$\hat y=\beta_1x_1+\beta_2x_2=(\beta_1+\beta_2)x.$$

The coefficients may be positive, zero or negative. We compare these three proposed allocations:

| Candidate | First coefficient | Second coefficient |
|:--|--:|--:|
| A | 1 | 0 |
| B | 1/2 | 1/2 |
| C | 0 | 1 |

Predict whether their fitted responses differ. Can squared prediction error alone decide which of the identical predictors is more important?

**Your prediction:** _Write here before evaluating the penalties._

For two coefficients, the objectives are

$$J_R=D+\lambda(\beta_1^2+\beta_2^2),\qquad
J_L=D+\lambda(|\beta_1|+|\beta_2|),\qquad \lambda=\frac43.$$

Complete the comparison. Explain which allocation Ridge favours in this table and whether Lasso distinguishes the three candidates.

| Candidate | Data-fit term D | Ridge objective | Lasso objective |
|:--|--:|--:|--:|
| A | … | … | … |
| B | … | … | … |
| C | … | … | … |

<details><summary>Hint</summary>

First compare the sums of the coefficients. Then compare the sum of their squares with the sum of their absolute values. These are different quantities.

</details>

**Your comparison:** _Write here._


### Are tied candidates actually optimal?

Equal objective values alone do not prove that several candidates minimise the objective. A better, unlisted pair might exist.

For this dataset, the Lasso objective can be written exactly as

$$J_L=2+\frac23(s-1)^2+
\frac43\bigl(|\beta_1|+|\beta_2|-s\bigr),
\qquad s=\beta_1+\beta_2.$$

Use this identity; you do not need to derive it. The last bracket is nonnegative because $|\beta_j|\geq\beta_j$ for each coefficient.

What is the smallest value this objective could attain? Check whether A, B and C attain it. Then find a fourth pair that attains the same value. What must be true of its sum and signs?

Explain whether a solver returning A would establish that the second predictor contains no useful information.

**Your reasoning:** _Distinguish an optimal allocation from the only optimal allocation._


## 3. Change only the units

Replace the second predictor by twice its previous value: $x_2'=2x$. Keep the responses, zero intercept and numerical penalty strength unchanged. Do not standardise the columns in this experiment.

Start from candidate B, $(1/2,1/2)$. In the new representation the prediction is

$$\hat y=\gamma_1x+\gamma_2(2x).$$

Keep $\gamma_1=1/2$. Before calculating penalties, predict the value of $\gamma_2$ that preserves every prediction. Would preserving predictions also preserve the penalised objective?

**Your prediction:** _Write here._

Find the converted coefficient pair. Compare its squared-coefficient sum and absolute-coefficient sum with those of B.

Finally compare the converted pair with $(0,1/2)$ in the **new** representation. Do these two pairs make the same predictions? Which has the smaller Lasso penalty?

Explain why a zero coefficient can depend on the representation, even when no information was added or removed. This is a comparison of equivalent predictions; it is not a derivation of the optimal refitted coefficients.

**Your comparison:** _Write here._


---

## 4. Verify with Python

Complete four expressions in the candidate loop: predictions, data-fit term, Ridge objective and Lasso objective. Use the definitions above.

The supplied plot varies the allocation continuously while keeping the coefficient sum at 1. Use your identity from Section 2 to explain why equal values on that line are genuine Lasso minima. The final calculation checks the unit conversion without refitting a model.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

x = np.array([-1, 0, 1], dtype=float)
y = 2 * x
lam = 4 / 3
coefficients = np.array([[1, 0], [0.5, 0.5], [0, 1]])


In [ ]:
ridge_values = []
lasso_values = []

for beta in coefficients:
    predictions = None  # ADD YOUR CODE HERE
    if predictions is None:
        break
    data_fit = None  # ADD YOUR CODE HERE
    ridge = None  # ADD YOUR CODE HERE
    lasso = None  # ADD YOUR CODE HERE
    if data_fit is None or ridge is None or lasso is None:
        break
    ridge_values.append(ridge)
    lasso_values.append(lasso)
    print('Coefficients:', beta, '| predictions:', predictions)
    print('Data fit, Ridge, Lasso:', round(data_fit, 3), round(ridge, 3), round(lasso, 3))


In [ ]:
if len(ridge_values) != 3 or len(lasso_values) != 3:
    print('Complete the four expressions, then rerun the comparison and this cell.')
else:
    first = np.linspace(0, 1, 101)
    second = 1 - first
    fixed_data_fit = np.mean((y - x) ** 2)
    ridge_curve = fixed_data_fit + lam * (first ** 2 + second ** 2)
    lasso_curve = fixed_data_fit + lam * (np.abs(first) + np.abs(second))

    plt.figure(figsize=(8, 4.5))
    plt.plot(first, ridge_curve, color='blue', label='Ridge')
    plt.plot(first, lasso_curve, color='red', label='Lasso')
    plt.scatter(coefficients[:, 0], ridge_values, color='blue', zorder=3)
    plt.scatter(coefficients[:, 0], lasso_values, color='red', zorder=3)
    plt.xlabel('First coefficient (second coefficient = 1 - first)')
    plt.ylabel('Objective value')
    plt.title('Different allocations, the same fitted responses')
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()

    original = np.array([0.5, 0.5])
    converted = np.array([0.5, 0.25])
    alternative = np.array([0, 0.5])
    original_prediction = original[0] * x + original[1] * x
    converted_prediction = converted[0] * x + converted[1] * (2 * x)
    alternative_prediction = alternative[0] * x + alternative[1] * (2 * x)
    print('Original, converted and alternative predictions:')
    print(original_prediction, converted_prediction, alternative_prediction)
    for beta in [original, converted, alternative]:
        print('Coefficients:', beta, '| squared sum:', np.sum(beta ** 2),
              '| absolute sum:', np.sum(np.abs(beta)))


---

## 5. Interpret the coefficients

1. Why can a smaller Ridge objective come with a larger training MSE? What would you still need to judge performance on new data?
2. If Lasso returns a zero coefficient for one of the identical predictors, what does the full set of minimisers tell you about that result?
3. Why does halving a coefficient after doubling its feature preserve predictions but change its penalty? What might therefore change when the model is refitted?

**Your answers:** _Use one or two sentences for each._

## Optional challenge

Replace the identical second column by one that is only approximately equal to the first. Which equality in your argument would no longer hold exactly? Explain why your exact nonuniqueness conclusion would then need to be checked again.

---

## Takeaway

Complete: **“To interpret a coefficient, I need to consider the information in the predictors, the penalty, and …”** Support your answer with the unit change.
